# ตอนที่ 3: RLHF และ PPO — เทรนโมเดลด้วยรางวัลที่หาอนุพันธ์ไม่ได้

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/03_rlhf_ppo.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 3/10] RLHF และ PPO: เทรนโมเดลด้วยรางวัลที่หาอนุพันธ์ไม่ได้**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-03-rlhf-ppo)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation)
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline ก่อนแตะอะไรทั้งสิ้น
6. เตรียมข้อมูล (Data)
7. โค้ดหลัก (Main code) — Stage A เทรน reward model จริง + Stage B **เขียน PPO เองจากศูนย์ ~120 บรรทัด**
8. ผลลัพธ์ (Results) — รัน PPO ใต้สายจูง KL ($\beta = 0.05$)
9. เปรียบเทียบ (Comparison) — **ถอดสายจูง KL ออก ($\beta = 0$) แล้วจับ reward hacking คาหนังคาเขา**
10. สรุป (Summary)

---

## หัวใจของโน้ตบุ๊กนี้

เราจะ **ไม่ใช้ `PPOTrainer` ของ TRL** — TRL ย้ายมันไปอยู่ `trl.experimental` แล้ว
และประกาศแผนถอดออก โค้ดที่สอนด้วยไลบรารีตัวนั้นจะรันไม่ได้ในอีกไม่กี่เดือน
ส่วนลูป PPO ที่เขียนเองราว 120 บรรทัดจะรันได้ตราบเท่าที่ PyTorch ยังอยู่

แต่ "เขียนเอง" ไม่ได้แปลว่า "ขอให้เชื่อ" — ชิ้นส่วนคณิตศาสตร์ทุกชิ้นถูกตรวจกับ
การคำนวณด้วยมือก่อนใช้จริง:

```python
assert torch.allclose(compute_gae(rewards, values), hand_math)   # GAE ตรงกับมือ
assert torch.allclose(surr, hand_clip)                           # clip ตรงกับมือ
```

และการทดลองปิดท้ายคือการ **ถอดสายจูง KL ออก** ($\beta = 0$) โดยแตะอย่างอื่นเลยแม้แต่ตัวเดียว
เพื่อวัด reward hacking ออกมาเป็นกราฟและตัวอย่างจริง ไม่ใช่คำกล่าวอ้าง

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16`, `attn_implementation="sdpa"` เอง
  อย่างชัดเจนทุกครั้ง (รายละเอียดอยู่ในคอมเมนต์ของ Cell 0 — อ่านให้ครบ)
  และสำหรับตอนนี้ fp16 มีระเบิดเพิ่มอีกลูกชื่อ **ratio overflow** — รอดูในหัวข้อ 7.2
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง
- **ห้ามเทรนบน KobEval-TH เด็ดขาด** ชุดนี้เป็นชุดวัดผลอย่างเดียว
  ข้อมูลที่ใช้เทรนในตอนนี้มาจาก `iapp/dpo_thai_tutorial` (Stage A) และ
  `VISAI-AI/gsm8k-thai` (Stage B) ซึ่งแยกขาดจากชุดวัดผล
- policy ตั้งต้นคือ **LoRA adapter จากตอนที่ 2** (`kobkrit/qwen3-0.6b-th-sft-lora`)
  ถ้าดึงจาก Hub ไม่ได้ โน้ตบุ๊กจะสร้าง adapter ใหม่ให้อัตโนมัติ (config เดิมของตอนที่ 2)
  แล้วพิมพ์บอกว่าใช้เส้นทางไหน — รันจบได้ด้วยตัวเองเสมอ

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลด policy + วัด baseline (TH-MATH 30 ข้อ) | ~4 นาที |
| Stage A: เทรน + วัด reward model | ~4 นาที |
| Stage B: เตรียมข้อมูล + self-check คณิตศาสตร์ | ~1 นาที |
| PPO รอบที่ 1 (β = 0.05) | ~5 นาที |
| PPO รอบที่ 2 (β = 0, ablation) | ~5 นาที |
| วัดผลซ้ำ + กราฟ + export | ~3 นาที |
| **รวม** | **~22-24 นาที** |

ถ้า T4 ที่ได้ช้ากว่าปกติ ตั้ง `FAST_MODE = True` ใน Cell 1 —
จะลดจำนวน prompt ของ PPO เหลือ 32 และลดความยาว generation ลง (~15 นาที)
โน้ตบุ๊กเล่าเรื่องเดิมได้ครบ เพียงแต่ขนาดของผลจะเล็กลง


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
REPO_DIR = "/content/thai-llm-tutorials"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

# ลดการแตกกระจายของหน่วยความจำ (fragmentation) -- ต้องตั้งก่อน torch แตะ CUDA
# T4 มี VRAM จำกัด พอ tensor ก้อนใหญ่ถูกจอง/คืนสลับกัน จะเกิดช่องว่างที่ใช้ต่อไม่ได้
# จน OOM ทั้งที่ยังเหลือที่รวม ๆ อยู่ (ข้อความ error ของ PyTorch เองก็แนะนำค่านี้)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# bf16 "แบบมีฮาร์ดแวร์รองรับจริง" เริ่มที่ Ampere (sm_80) ขึ้นไปเท่านั้น
#
# หมายเหตุสำคัญ: torch.cuda.is_bf16_supported() ตอบ True บน T4 (sm_75) ด้วย
# ตั้งแต่ torch รุ่นใหม่ ๆ เพราะมันนับ "การจำลอง (emulation)" ว่ารองรับด้วย
# ซึ่งช้ากว่า fp16 มากและไม่ใช่สิ่งที่เราต้องการ เราจึงเช็ค compute capability
# ตรง ๆ แทน -- นี่คือกับดักจริงที่เจอตอนรันบน Colab
NATIVE_BF16 = CAPABILITY[0] >= 8
SUPPORTS_BF16 = NATIVE_BF16          # ใช้ชื่อเดิมต่อได้ แต่ความหมายคือ "native"

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}   (native; torch บอกว่า "
      f"{torch.cuda.is_bf16_supported()} เพราะนับ emulation ด้วย)")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False   (native)
# ทั้งที่ torch.cuda.is_bf16_supported() ตอบ True -- อย่าเชื่อค่านั้น
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
#    pip install -e ลงทะเบียนแพ็กเกจไว้แล้ว แต่ sys.path ของเคอร์เนลที่รันอยู่
#    อาจยังไม่เห็น จึงใส่ path ของ repo เข้าไปตรง ๆ กันพลาด (idempotent)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums
from kobeval.plotting import use_thai_font

# 8) ตั้งฟอนต์ไทยให้ matplotlib "ครั้งเดียวตรงนี้" มีผลกับทุกกราฟในโน้ตบุ๊ก
#    ทำไมต้องมีขั้นตอนนี้: apt ติดตั้งฟอนต์ใน (2) หลังจาก matplotlib สร้าง
#    แคชรายชื่อฟอนต์ไปแล้ว มันจึงมองไม่เห็นฟอนต์ไทย และวาดออกมาเป็นกล่อง []
#    use_thai_font() จะสแกนดิสก์ใหม่แล้วลงทะเบียนฟอนต์ให้
_thai_font = use_thai_font()
print(f"Thai font      : {_thai_font or 'ไม่พบ! กราฟภาษาไทยจะเป็นกล่อง [] -- รัน apt-get ในข้อ (2) ใหม่'}")

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")


---

## 1. ปัญหา (Problem statement)

SFT ในตอนที่ 2 มีสมมติฐานซ่อนอยู่หนึ่งข้อ: **ต้องมีเฉลยให้เลียนแบบ**
แต่สิ่งที่เราอยากได้จริง เช่น *"ตอบโจทย์เลขให้ถูก และอธิบายเป็นภาษาไทยที่อ่านรู้เรื่อง"*
ไม่มีเฉลยเดียว คำตอบที่ดีมีได้ร้อยแบบ และคำว่า "อ่านรู้เรื่อง" ก็เขียนเป็นสมการไม่ออก

พอพยายามจะ optimize สิ่งเหล่านี้ตรง ๆ เราจะชนกำแพงสองชั้นเสมอ:

| ทางที่อยากเดิน | ติดกำแพงอะไร |
|---|---|
| เขียน loss ของ "คำตอบที่ดี" ตรง ๆ | นิยาม "ดี" เป็นสมการไม่ได้ มีแต่การ **เปรียบเทียบ** |
| ให้คนให้คะแนน แล้ว backprop | คะแนนอยู่ *หลัง* การสุ่ม token — gradient เดินย้อนผ่านการสุ่มไม่ได้ |
| ให้คนนั่งให้คะแนนสดระหว่างเทรน | มนุษย์ให้คะแนนไม่ทันแม้แต่เสี้ยวเดียวของ rollout |

นี่คือที่มาของชื่อตอน: เรากำลังจะ optimize **รางวัลที่หาอนุพันธ์ไม่ได้**
เครื่องมือที่ทำแบบนั้นได้ชื่อว่า reinforcement learning

---

## 2. เราจะทำอะไร (Solution)

RLHF แก้กำแพงทั้งสองชั้นด้วยการเดินสองขั้น:

- **Stage A — Reward Model:** เทรน $r_\phi(x,y)$ ให้เลียนแบบการเปรียบเทียบของมนุษย์
  จากคู่ preference ภาษาไทย 100 คู่ (`iapp/dpo_thai_tutorial`) — แก้กำแพงที่หนึ่ง
- **Stage B — PPO:** เขียนลูป PPO เอง **จากศูนย์ราว 120 บรรทัด** แล้วดัน policy
  เข้าหารางวัลที่ตรวจได้ด้วยกติกา บนโจทย์เลขไทยจาก `VISAI-AI/gsm8k-thai`
  พร้อม **สายจูง KL** รั้งไม่ให้วิ่งหนีโมเดลตั้งต้น — แก้กำแพงที่สอง

ราคาที่จ่ายคือความซับซ้อน: ระหว่างเทรนจะมีโมเดล **สี่บทบาท** อยู่พร้อมกัน —
policy $\pi_\theta$ (ตัวที่เทรน), reference $\pi_{\text{ref}}$ (ตัวตั้งต้นที่แช่แข็ง),
reward model $r_\phi$ และ value network $V_\psi$

บน T4 ฟรีเรารอดเพราะบทเรียนจากตอนที่ 2: policy = base + **LoRA adapter จากตอนที่ 2**
ส่วน reference คือ base ตัวเดิมที่ **ปิด adapter** (`policy.disable_adapter()`) —
ต้นทุน VRAM ของ reference จึงเป็น **ศูนย์ไบต์** ท่าเดียวกับที่ตอนที่ 4 จะใช้กับ DPO

> **แนวคิดหลักของตอนนี้:** RLHF คือการ optimize รางวัลที่หาอนุพันธ์ไม่ได้
> ผ่านตัวแทนที่ไม่สมบูรณ์ (reward model หรือ rule) และการ optimize ตัวแทนแรง ๆ
> โดยไม่มีอะไรรั้ง จะพังตาม **Goodhart's law** เสมอ:
> เมื่อตัวชี้วัดกลายเป็นเป้าหมาย มันจะเลิกเป็นตัวชี้วัดที่ดี
>
> พจน์ KL ในสมการ 3.2 จึง **ไม่ใช่ regularizer** ที่ใส่ไว้กันเหนียว —
> มันคือ **สิ่งเดียว** ที่ขวางอยู่ระหว่างคุณกับ reward hacking
> หัวข้อ 9 จะพิสูจน์ประโยคนี้ด้วยการถอดมันออกให้ดูกับตา

และอีกประโยคที่ควรถือไว้ทั้งซีรีส์: objective ของตอนนี้คือ **สมการแม่ของครึ่งหลังของซีรีส์**
ตอนที่ 4 (DPO) แก้สมการนี้ในรูปปิดจน reward model และ RL loop ตัดกันหายไป
ตอนที่ 5 (GRPO) เปลี่ยนวิธีประมาณ advantage จน value network หายไป


---

## 3. สมการ (Equation)

### 3.1 Reward model: Bradley–Terry

$$
\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\Big[\log\sigma\big(r_\phi(x,y_w) - r_\phi(x,y_l)\big)\Big]
$$

$r_\phi(x,y)$ คือคะแนนสเกลาร์หนึ่งตัวต่อหนึ่งข้อความ — ในทางปฏิบัติคือโมเดลภาษา
ที่เปลี่ยนหัวเป็น linear ชั้นเดียว (`num_labels=1`)

จุดที่คนมองข้ามแล้วไปเจ็บตัวทีหลัง: loss นี้เห็นแค่ **ผลต่าง** ของคะแนน
แทน $r_\phi$ ด้วย $r_\phi + c$ ด้วยค่าคงที่ใดก็ได้ — loss ไม่เปลี่ยนเลย
แปลว่า **สเกลสัมบูรณ์ของ reward ไม่ถูกนิยามโดยการเทรน** รันสองครั้งอาจได้คะแนนเฉลี่ย
3.7 กับ −12.4 ที่จัดอันดับเหมือนกันเป๊ะ นี่คือเหตุผลที่ต้อง **standardize reward
ก่อนป้อนเข้า PPO เสมอ** (ลบ mean หารด้วย std) — จำจุดนี้ไว้ มันจะกลับมาในหัวข้อ 7.2

### 3.2 สมการแม่: RLHF objective

$$
\max_\theta\ \mathbb{E}_{x\sim\mathcal{D},\,y\sim\pi_\theta(\cdot|x)}\big[r_\phi(x,y)\big] \;-\; \beta\,\mathbb{D}_{\text{KL}}\big(\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big)
$$

อ่านเป็นภาษาคน: **"เก็บคะแนน reward ให้มากที่สุด แต่ทุกก้าวที่เดินห่างจากโมเดลตั้งต้น
ต้องจ่ายค่าปรับ"** — $\beta$ คือราคาต่อหนึ่ง nat ของการเดินห่าง คือความตึงของสายจูง
สังเกตว่า $y$ ถูก **สุ่มจาก $\pi_\theta$ เอง** นี่คือความต่างเชิงโครงสร้างจาก SFT/DPO
ที่เรียนจากข้อมูลนิ่ง ๆ ในไฟล์

### 3.3 PPO clipped surrogate: เครื่องยนต์ของ Stage B

การ generate คือคอขวด PPO จึงอยากรีดค่า rollout ชุดเดิมหลาย epoch
ต้องมีตัวคูณแก้ทาง (importance sampling ratio) แล้วหนีบมันไว้:

$$
\rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)},\qquad
\mathcal{L}^{\text{CLIP}} = \mathbb{E}_t\Big[\min\big(\rho_t\,\hat A_t,\ \text{clip}(\rho_t,\,1-\epsilon,\,1+\epsilon)\,\hat A_t\big)\Big]
$$

หัวใจคือ **min + clip ทำงานร่วมกันแบบมองโลกแง่ร้ายอย่างจงใจ**:
$\hat A_t > 0$ ผลตอบแทนถูกตัดเพดานที่ $1+\epsilon$ แต่ $\hat A_t < 0$ ตัว min
เลือกฝั่งที่แย่กว่าเสมอ — **ได้จำกัด เสียไม่จำกัด** นโยบายจึงขยับทีละก้าวเล็ก ๆ

> **อย่าสับสน: มี "โมเดลเก่า" สองตัว และมันคนละตัวกัน**
> $\pi_{\text{ref}}$ (สมการ 3.2) แช่แข็ง **ตลอดการเทรน** ทำหน้าที่สายจูง KL
> $\pi_{\theta_{\text{old}}}$ (สมการ 3.3) คือ snapshot **ณ ตอน rollout ล่าสุด** เปลี่ยนทุกรอบ
> ทำหน้าที่ trust region — บั๊กยอดฮิตอันดับหนึ่งของคนเขียน PPO เองคือจับสองตัวนี้
> ใส่ตัวแปรเดียวกัน

### 3.4 GAE: คำนวณ advantage อย่างไรไม่ให้จมน้ำเสียง noise

$$
\delta_t = r_t + \gamma V_\psi(s_{t+1}) - V_\psi(s_t),\qquad
\hat A_t = \sum_{l=0}^{\infty} (\gamma\lambda)^l\,\delta_{t+l}
$$

$V_\psi$ ทำนายว่า "จากจุดนี้ไปจนจบ จะเก็บ reward ได้อีกเท่าไหร่" — นี่คือ **โมเดลตัวที่สี่**
$\lambda$ คือปุ่ม bias–variance: $\lambda=0$ เชื่อ $V_\psi$ สุดใจ, $\lambda=1$ รอดูผลจริง
พร้อมแบก noise ทั้งสาย ค่าที่นิยมคือ **0.95** (เราใช้ $\gamma = 1.0$ ตามธรรมเนียมงาน LLM)

> จำ $V_\psi$ ตัวนี้ไว้ให้ดี — ตอนที่ 5 (GRPO) จะฆ่ามันทิ้งด้วยค่าเฉลี่ยของกลุ่มคำตอบ

### 3.5 Loss เต็มของ PPO: สามพจน์

$$
\mathcal{L}_{\text{PPO}} = -\mathcal{L}^{\text{CLIP}} + c_1\,\mathbb{E}_t\big[(V_\psi(s_t) - \hat R_t)^2\big] - c_2\,\mathbb{E}_t\big[\mathcal{H}[\pi_\theta(\cdot|s_t)]\big]
$$

$c_1 = 0.5$, $c_2 = 0.01$ ส่วนสายจูง KL ของสมการ 3.2 นิยมยัดเข้าไปใน reward ต่อ token:
$r_t \leftarrow r_t - \beta(\log\pi_\theta - \log\pi_{\text{ref}})$ ซึ่งเป็นวิธีที่เราใช้ในหัวข้อ 7.2

นับของที่ต้องจูน: โมเดล 4 บทบาท บวก $\epsilon, \beta, \gamma, \lambda, c_1, c_2$ และ
learning rate อีกสองชุด — นี่คือเหตุผลที่ PPO ขึ้นชื่อว่า "รันสองรอบด้วย seed ต่างกัน
ได้ผลคนละเรื่อง" และเป็นเหตุผลการมีอยู่ของตอนที่ 4 ทั้งตอน

---

## 4. เห็นภาพสมการ (Visualize)

รูปของหัวข้อนี้ถูกวาดในเซลล์ **ถัดจาก Cell 1** เพราะกติกาของซีรีส์กำหนดว่า
เซลล์โค้ดที่สองของทุกตอนต้องเป็นการวัด baseline เสมอ ห้ามมีอะไรมาแทรกก่อน

กราฟจะแสดงสองภาพ: ความไม่สมมาตรของ clipped surrogate (สมการ 3.3)
และปุ่ม bias–variance ของ GAE (สมการ 3.4) บน rollout สังเคราะห์


---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 ซึ่งประกอบ policy ตั้งต้น (base + LoRA adapter จากตอนที่ 2
ถ้าดึงได้ ไม่ได้ก็สร้างใหม่) แล้ววัดด้วย KobEval-TH ก่อนที่เราจะเทรนอะไรทั้งนั้น
ตัวเลขชุดนี้คือแถว "ตั้งต้น" ของตารางในหัวข้อ 9

> **สเกลจริงของสิ่งที่เรากำลังย่อส่วน — อ่านก่อนรัน**
> RLHF ระดับใช้งานจริงถือโมเดล 4 ตัวที่ตัวใหญ่สุดมัก 7B ขึ้นไป ใช้คู่ preference
> จากมนุษย์จริง **หลักหมื่นถึงหลักล้านคู่** และแยกเครื่อง generate ออกจากเครื่องเทรน
> โน้ตบุ๊กนี้ใช้ Qwen3-0.6B ทุกตำแหน่ง คู่ preference 100 คู่ และโจทย์เลข 64 ข้อ
> สิ่งที่มันสาธิตคือ **อัลกอริทึมครบทุกชิ้นส่วน** — ไม่ใช่การทำ RLHF จริง

> **หมายเหตุเรื่องเวลา:** เรารัน KobEval-TH เฉพาะ slice `TH-MATH` (30 ข้อ)
> เพราะ reward ของ Stage B คือความถูกต้องของโจทย์เลขไทย ซึ่ง TH-MATH วัดตรงที่สุด
> สังเกต `th_ratio` ด้วย — โบนัส +0.2 ของ reward ผูกกับมันตรง ๆ
> เพิ่ม slice อื่นใน `KOBEVAL_SLICES` ได้ถ้ามีเวลา — สัญญาการวัดผลเหมือนกันทุกประการ


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนเทรน/ก่อนแก้อะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
import gc
import json
import math
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "Qwen/Qwen3-0.6B"
SFT_ADAPTER_ID = "kobkrit/qwen3-0.6b-th-sft-lora"   # adapter จากตอนที่ 2
KOBEVAL_SLICES = ["TH-MATH"]    # reward ของตอนนี้คือโจทย์เลขไทย -- slice นี้วัดตรงที่สุด
DEV = "cuda:0"
FAST_MODE = False               # True = ลด prompt เหลือ 32 + generation สั้นลง ถ้า T4 ช้า

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)

cfg = model.config
print(f"โหลดโมเดลฐานแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"  hidden_size={cfg.hidden_size}  layers={cfg.num_hidden_layers}  "
      f"heads={cfg.num_attention_heads}/{cfg.num_key_value_heads} (GQA)  "
      f"vocab={cfg.vocab_size}")

# --- ประกอบ policy: base + LoRA adapter จากตอนที่ 2 (ถ้าดึงได้) --------------
# ตอนที่ 2 จบด้วยการ push adapter ขึ้น Hub -- ถ้าคุณยังไม่ได้รันตอนที่ 2 หรือ
# adapter ยังไม่ถูก push โน้ตบุ๊กนี้จะสร้าง adapter ใหม่ด้วย config เดิมเป๊ะ
# (r=16, alpha=32, ครบ 7 เมทริกซ์) ซึ่ง B=0 ทำให้ policy ตั้งต้น == base พอดี
try:
    policy = PeftModel.from_pretrained(model, SFT_ADAPTER_ID, is_trainable=True)
    POLICY_SOURCE = SFT_ADAPTER_ID
    print(f"\npolicy = base + adapter จากตอนที่ 2 ({SFT_ADAPTER_ID})")
except Exception as exc:
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,   # PPO ต้อง deterministic: dropout ทำให้ ratio != 1 ตั้งแต่ epoch แรก
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    policy = get_peft_model(model, lora_cfg)
    POLICY_SOURCE = "fresh-lora (fallback)"
    print(f"\nดึง {SFT_ADAPTER_ID} ไม่ได้ ({type(exc).__name__}) -> สร้าง adapter ใหม่")
    print("โน้ตบุ๊กยังรันจบได้ครบทุกเซลล์ เพียงแต่ policy ตั้งต้นคือ base ล้วน (B=0)")


def cast_trainable_to_fp32(m):
    """กับดัก fp16 ของซีรีส์: cast 'เฉพาะพารามิเตอร์ที่เทรนได้' เป็น fp32"""
    n_cast = 0
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
            n_cast += 1
    return n_cast


n_cast = cast_trainable_to_fp32(policy)
n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"พารามิเตอร์ที่เทรนได้ของ policy: {n_trainable / 1e6:.2f}M "
      f"(cast เป็น fp32 ไป {n_cast} tensors)")

VRAM_AFTER_POLICY = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"VRAM หลังประกอบ policy: {VRAM_AFTER_POLICY:.3f} GB")

policy.eval()

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
t0 = time.time()
baseline = evaluate(
    policy,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name=f"policy ตั้งต้น ({POLICY_SOURCE})",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัด baseline: {time.time() - t0:.0f} วินาที")

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

print(f"\nรวมทุก slice: acc={baseline['overall']['accuracy']:.3f}  "
      f"th_ratio={baseline['overall']['th_ratio']:.2f}")


---

### รูปประกอบหัวข้อที่ 4 — ความไม่สมมาตรของ clip และปุ่ม λ ของ GAE

เซลล์ถัดไปไม่ใช้ GPU และไม่ใช้เน็ต วาดจากสมการ 3.3 และ 3.4 ตรง ๆ

- **ภาพซ้าย:** ฝั่ง $\hat A > 0$ ผลตอบแทนแบนสนิทหลัง $1+\epsilon$ (gradient เป็นศูนย์ —
  "เรียนจบแล้ว") แต่ฝั่ง $\hat A < 0$ ค่าปรับไม่มีเพดาน เพราะ min เลือกกิ่งที่แย่กว่าเสมอ
- **ภาพขวา:** rollout สังเคราะห์ 20 ขั้น ($\gamma = 1$): reward ต่อขั้นเป็น noise เล็ก ๆ
  คะแนนจริงมาตอนจบ และ $V_\psi$ ตั้งใจให้ทายต่ำกว่าจริง — ที่ $\lambda = 0$ สัญญาณ
  ไปไม่ถึง token ต้น ๆ, ที่ $\lambda = 1$ ทุก token ได้เครดิตเต็มพร้อม noise เต็ม
  (ภาพประกอบกลไก ไม่ใช่ข้อมูลเทรนจริง)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 -- clipped surrogate (สมการ 3.3) และ GAE ที่ lambda ต่างกัน (สมการ 3.4)
# เซลล์นี้ไม่ใช้ GPU และไม่ใช้เน็ต
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt

from kobeval import use_thai_font

use_thai_font()

EPS_PLOT = 0.2
rho = np.linspace(0.0, 2.0, 400)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))

for A, color, label in [(+1.0, "#16a34a", "Â = +1 (token ดี)"),
                        (-1.0, "#dc2626", "Â = −1 (token แย่)")]:
    surr = np.minimum(rho * A, np.clip(rho, 1 - EPS_PLOT, 1 + EPS_PLOT) * A)
    axes[0].plot(rho, surr, lw=2, color=color, label=label)
axes[0].axvline(1 - EPS_PLOT, ls=":", color="#64748b", lw=1)
axes[0].axvline(1 + EPS_PLOT, ls=":", color="#64748b", lw=1)
axes[0].axvspan(1 + EPS_PLOT, 2.0, alpha=0.08, color="#16a34a")
axes[0].text(1.62, 1.05, "แบน = gradient ศูนย์", fontsize=8, color="#475569")
axes[0].set_xlabel("probability ratio ρ")
axes[0].set_ylabel("clipped surrogate")
axes[0].set_title("ได้จำกัด (เพดาน 1+ε) แต่เสียไม่จำกัด", fontsize=11)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_axisbelow(True)

# --- GAE บน rollout สังเคราะห์ 20 ขั้น -----------------------------------------
T_STEPS = 20
rng = np.random.default_rng(SEED)
rewards_syn = rng.normal(0.0, 0.05, size=T_STEPS)
rewards_syn[-1] += 1.0                       # คะแนนจริงมาตอนจบ (เหมือน task reward ของเรา)
true_v = np.array([1.0] * T_STEPS + [0.0])   # มูลค่าจริงคร่าว ๆ (gamma=1)
values_syn = true_v - 0.4                    # V ตั้งใจทายต่ำกว่าจริง 0.4
values_syn[-1] = 0.0                         # terminal


def gae_np(rewards, values, gamma, lam):
    adv, acc = np.zeros_like(rewards), 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        acc = delta + gamma * lam * acc
        adv[t] = acc
    return adv


for lam, color in [(0.0, "#16a34a"), (0.5, "#f59e0b"), (0.95, "#2563eb"), (1.0, "#dc2626")]:
    adv = gae_np(rewards_syn, values_syn, gamma=1.0, lam=lam)
    axes[1].plot(np.arange(T_STEPS), adv, lw=2, color=color, label=f"λ = {lam}")
axes[1].axhline(0, color="#94a3b8", lw=1, ls=":")
axes[1].set_xlabel("ตำแหน่ง token t ใน rollout")
axes[1].set_ylabel("advantage Â_t")
axes[1].set_title("λ ต่ำ = สัญญาณไปไม่ถึงต้นสาย | λ สูง = แบก noise เต็ม", fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)

fig.tight_layout()
fig.savefig("fig_ppo_clip_gae.png", dpi=150, bbox_inches="tight")
plt.show()

print("บันทึกรูปแล้ว: fig_ppo_clip_gae.png")
print("อ่านภาพขวา: ที่ lambda=0.95 คะแนนตอนจบเดินทางถึง token ต้น ๆ โดย noise ถูกหน่วง")
print("นี่คือเหตุผลที่ทั้งวงการใช้ 0.95 และเราก็จะใช้ค่านี้ในหัวข้อ 7.2")


---

## 6. เตรียมข้อมูล (Data)

### Stage A — คู่ preference สำหรับ reward model

ใช้ `iapp/dpo_thai_tutorial` (100 คู่, Apache-2.0) — ชุดข้อมูล preference ภาษาไทย
ที่ผมทำขึ้นเองสำหรับซีรีส์นี้ แต่ละแถวมี `prompt`, `chosen`, `rejected` ที่คัดด้วยมือ

แบ่ง **80/20**: เทรน 80 คู่ กัน 20 คู่ไว้เป็น held-out **ห้ามแตะระหว่างเทรน**
เกณฑ์ผ่านของ Stage A คือ pairwise ranking accuracy บน 20 คู่ที่กันไว้ พร้อม
**Wilson 95% CI ที่ไม่คร่อม 0.5** — บน 20 คู่ ต้องถูกอย่างน้อย 15/20 ถึงจะพูดแบบนั้นได้
(`wilson_ci(15, 20)` เริ่มที่ 0.531 ส่วน `wilson_ci(14, 20)` ยังเริ่มที่ 0.481)
ถ้าไม่ถึง เราจะพูดตามที่ตัวเลขบอก: ยังแยกจากการโยนเหรียญไม่ได้

### Stage B — โจทย์ที่ตรวจได้ด้วยกติกา

ลูป PPO ใช้โจทย์เลข 64 ข้อจาก `VISAI-AI/gsm8k-thai` (GSM8K ฉบับแปลไทย)
และให้คะแนนด้วย **กติกาที่ตรวจสอบได้**:

- **+1.0** ถ้าเลขจำนวนเต็ม *ตัวสุดท้าย* ในคำตอบตรงกับเฉลย —
  โดย normalize เลขไทย **๐-๙** เป็นอารบิกก่อนเสมอ (ด้วย `thai_digits_to_arabic`
  ของ kobeval — เราไม่เขียน metric ซ้ำเอง)
- **+0.2** ถ้าคำตอบเป็นภาษาไทยจริง (`th_ratio >= 0.5` — ฟังก์ชันเดียวกับที่
  KobEval-TH ใช้ตัดสิน ไม่ใช่ heuristic ใหม่)

> **ทำไม Stage B ไม่ใช้ reward model จาก Stage A**
> ในระบบจริง Stage B กินผลผลิตของ Stage A ตรง ๆ — นั่นคือนิยามของ RLHF
> แต่ RM ที่เทรนจาก 100 คู่ **อ่อนเกินกว่าจะรับแรงกดดันของ PPO** มันจะโดน hack
> ภายในไม่กี่ update แล้วเราจะแยกไม่ออกว่าลูป PPO ผิด หรือ RM แค่อ่อน —
> การทดลองจะพิสูจน์อะไรไม่ได้เลย
>
> rule reward จึงทำหน้าที่เป็น **stand-in ของ RM** ที่ตรวจสอบได้: เมื่อ reward ขึ้น
> เรารู้แน่ว่าลูปทำงานจริง แต่มันยังคง "ไม่สมบูรณ์" เหมือน RM ทุกตัว — มันวัดแค่
> เลขท้ายกับสัดส่วนอักขระไทย ไม่วัดความอ่านรู้เรื่องของทุกอย่างรอบ ๆ
> ซึ่งเป็นช่องโหว่ที่การทดลอง $\beta = 0$ ในหัวข้อ 9 จะทิ่มให้ดู
> (แนวคิด reward ตรวจได้ด้วยกติกาจะกลับมาเป็นพระเอกเต็มตัวในตอนที่ 5)

เซลล์ถัดไปยังทดสอบ `rule_reward` กับเคสที่รู้คำตอบล่วงหน้า — รวมทั้งเลขไทย ๑๒๓ —
ก่อนปล่อยให้มันเป็นกรรมการของการเทรนทั้งตอน


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6 -- โหลดข้อมูลสองชุด + นิยาม rule reward + ทดสอบกรรมการก่อนใช้จริง
# -----------------------------------------------------------------------------
import re

from datasets import load_dataset
from kobeval import thai_digits_to_arabic


def pick_column(columns, candidates):
    """หาชื่อคอลัมน์แบบไม่แคร์ตัวพิมพ์ใหญ่เล็ก -- schema ของชุดข้อมูลสาธารณะเปลี่ยนได้"""
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


def to_gen_prompt(q):
    """prompt สำหรับ generate ตาม chat template ของ Qwen3 (ปิด thinking mode)"""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


def to_rm_text(q, a):
    """ข้อความเต็ม (คำถาม+คำตอบ) สำหรับให้ reward model ให้คะแนน"""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}, {"role": "assistant", "content": a}],
        tokenize=False, enable_thinking=False,
    )


# --- Stage A: iapp/dpo_thai_tutorial แบ่ง 80/20 -------------------------------
pref_ds = load_dataset("iapp/dpo_thai_tutorial", split="train")
print(f"iapp/dpo_thai_tutorial columns: {pref_ds.column_names}  n={len(pref_ds)}")

p_col = pick_column(pref_ds.column_names, ["prompt", "question", "instruction", "input"])
c_col = pick_column(pref_ds.column_names, ["chosen", "preferred", "good", "response_j"])
r_col = pick_column(pref_ds.column_names, ["rejected", "dispreferred", "bad", "response_k"])
assert None not in (p_col, c_col, r_col), f"schema ไม่ตรงที่คาด: {pref_ds.column_names}"

rm_pairs = []
for row in pref_ds:
    q, w, l = row[p_col], row[c_col], row[r_col]
    if not all(isinstance(v, str) and v.strip() for v in (q, w, l)):
        continue
    rm_pairs.append({"chosen": to_rm_text(q.strip(), w.strip()),
                     "rejected": to_rm_text(q.strip(), l.strip())})

random.Random(SEED).shuffle(rm_pairs)
n_rm_train = int(round(0.8 * len(rm_pairs)))
RM_TRAIN, RM_HELDOUT = rm_pairs[:n_rm_train], rm_pairs[n_rm_train:]
print(f"คู่ preference: {len(rm_pairs)} | เทรน {len(RM_TRAIN)} | held-out {len(RM_HELDOUT)}")

# --- Stage B: VISAI-AI/gsm8k-thai --------------------------------------------
NUM_RE = re.compile(r"-?\d+")


def last_int(text):
    """เลขจำนวนเต็มตัวสุดท้ายในข้อความ (normalize เลขไทย ๐-๙ ก่อนเสมอ)"""
    nums = NUM_RE.findall(thai_digits_to_arabic(text).replace(",", ""))
    return int(nums[-1]) if nums else None


gsm_all = load_dataset("VISAI-AI/gsm8k-thai")
gsm_split = "train" if "train" in gsm_all else list(gsm_all.keys())[0]
gsm = gsm_all[gsm_split]
print(f"gsm8k-thai[{gsm_split}] columns: {gsm.column_names}  n={len(gsm)}")

g_q = pick_column(gsm.column_names, ["translated_question", "question", "question_th", "instruction", "problem", "input", "query"])
g_a = pick_column(gsm.column_names, ["translated_answer", "answer", "answer_th", "solution", "output", "response"])
assert None not in (g_q, g_a), f"schema ไม่ตรงที่คาด: {gsm.column_names}"

N_PPO_PROMPTS = 32 if FAST_MODE else 64
N_PROBE = 6

gsm = gsm.shuffle(seed=SEED).select(range(min(N_PPO_PROMPTS + N_PROBE + 200, len(gsm))))
math_items = []
for row in gsm:
    q, a = row[g_q], row[g_a]
    if not (isinstance(q, str) and isinstance(a, str)):
        continue
    gold = last_int(a)   # เฉลย GSM8K จบด้วยเลขคำตอบเสมอ -> เลขตัวสุดท้ายคือ gold
    if gold is None or not (20 <= len(q.strip()) <= 600):
        continue
    math_items.append({"question": q.strip(), "gold": gold})
    if len(math_items) >= N_PPO_PROMPTS + N_PROBE:
        break

assert len(math_items) >= N_PPO_PROMPTS + N_PROBE, "ดึงโจทย์ได้ไม่พอ -- ขยาย range ด้านบน"
PROBE_ITEMS = math_items[:N_PROBE]              # held-out ไว้ส่องพฤติกรรม ห้ามเทรน
TRAIN_ITEMS = math_items[N_PROBE:N_PROBE + N_PPO_PROMPTS]
print(f"โจทย์เลขไทย: เทรน {len(TRAIN_ITEMS)} ข้อ | probe held-out {len(PROBE_ITEMS)} ข้อ")
print(f"ตัวอย่างโจทย์: {TRAIN_ITEMS[0]['question'][:140]}...")
print(f"เฉลย (gold) : {TRAIN_ITEMS[0]['gold']}")


# --- กรรมการของ Stage B: rule reward ------------------------------------------
def rule_reward(response, gold):
    """+1.0 ถ้าเลขจำนวนเต็มตัวสุดท้ายถูก (รองรับเลขไทย ๐-๙), +0.2 ถ้าตอบเป็นภาษาไทยจริง

    ใช้ thai_digits_to_arabic + th_ratio ของ kobeval ตรง ๆ -- ไม่เขียน metric ซ้ำเอง
    """
    val = last_int(response)
    correct = 1.0 if (val is not None and val == int(gold)) else 0.0
    thai_bonus = 0.2 if th_ratio(response) >= 0.5 else 0.0
    return correct + thai_bonus


# --- ทดสอบกรรมการก่อนใช้จริง (รวมเลขไทย ๐-๙) ----------------------------------
_CASES = [
    ("ดังนั้นคำตอบคือ ๑๒๓", 123, 1.2),   # เลขไทยถูก + เป็นภาษาไทย
    ("ดังนั้นคำตอบคือ 123", 123, 1.2),   # เลขอารบิกถูก + เป็นภาษาไทย
    ("The answer is 123", 123, 1.0),      # ถูกแต่ไม่ใช่ภาษาไทย -> ไม่ได้โบนัส
    ("ดังนั้นคำตอบคือ ๑๒๔", 123, 0.2),   # ผิด แต่เป็นภาษาไทย -> ได้แค่โบนัส
    ("คำตอบคือ 1,234 บาท", 1234, 1.2),   # comma คั่นหลักต้องไม่ทำให้พัง
    ("ไม่มีตัวเลขเลย", 5, 0.2),
    ("", 5, 0.0),
]
for resp, gold, want in _CASES:
    got = rule_reward(resp, gold)
    assert abs(got - want) < 1e-9, f"rule_reward({resp!r}, {gold}) = {got} คาด {want}"
    print(f"rule_reward({resp!r:32s} gold={gold:5d}) = {got:.1f}  ตรงที่คาด")
print("=> กรรมการผ่านการสอบทุกเคส รวมทั้งเลขไทย ๑๒๓ -- ใช้ตัดสินการเทรนได้")


---

## 7. โค้ดหลัก (Main code)

### 7.1 Stage A — เทรน reward model จริง

เปลี่ยนโมเดลภาษาให้เป็นเครื่องให้คะแนน: หัว LM ถูกแทนด้วย linear ชั้นเดียว
ที่คืนสเกลาร์หนึ่งตัว (`AutoModelForSequenceClassification`, `num_labels=1`)
แล้วเทรนด้วยสมการ 3.1 ตรงตัว: `-F.logsigmoid(s_w - s_l).mean()`

สี่รายละเอียดที่ทำให้เซลล์ถัดไปรันได้จริง ไม่ใช่แค่สวยบนกระดาน:

1. **`config.pad_token_id` ต้องถูกตั้ง** — หัว SeqCls ของ transformers เลือกคะแนน
   จาก token สุดท้ายที่ไม่ใช่ pad ถ้าไม่ตั้ง มันจะไม่รู้ว่า pad คือตัวไหน
   แล้วพังทันทีที่ batch มีความยาวไม่เท่ากัน
2. **LoRA r=8 เฉพาะ attention + `modules_to_save=["score"]`** — หัวคะแนนตั้งต้น
   แบบสุ่ม ต้องเทรนเต็มทั้งหัว จะติด adapter ทับหัวสุ่มอย่างเดียวไม่ได้
3. **cast เฉพาะพารามิเตอร์ที่เทรนได้เป็น fp32** — บทเรียน fp16 เดิมของซีรีส์
4. **GradScaler** — เทรนใน fp16 แบบ manual loop ต้องมี loss scaling เอง
   (`Trainer` เคยทำให้เราอัตโนมัติ ตอนนี้เราคือคนขับเอง)

เกณฑ์ตัดสินอยู่ท้ายเซลล์: pairwise accuracy บน 20 คู่ held-out พร้อม Wilson CI
และคำพิพากษาที่ซื่อสัตย์ว่า CI พ้น 0.5 หรือไม่


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.1 -- Stage A: เทรน reward model จากคู่ preference 80 คู่ แล้ววัดบน 20 คู่
# -----------------------------------------------------------------------------
from transformers import AutoModelForSequenceClassification

RM_EPOCHS = 2 if FAST_MODE else 3
RM_BATCH = 2          # คู่ต่อ batch (chosen 2 + rejected 2 = 4 ข้อความต่อ step)
RM_MAX_LEN = 512
RM_LR = 1e-4

rm = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=1,                  # หัวสเกลาร์: หนึ่งคะแนนต่อหนึ่งข้อความ (สมการ 3.1)
    torch_dtype=DTYPE,             # สัญญาเดิมของซีรีส์ -- ไม่พึ่ง auto
    attn_implementation=ATTN_IMPL,
    device_map=DEV,
)
rm.config.pad_token_id = tokenizer.pad_token_id   # จำเป็น! หัว SeqCls ต้องรู้ว่า pad คือตัวไหน

rm = get_peft_model(rm, LoraConfig(
    task_type="SEQ_CLS",
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],     # หัวคะแนนตั้งต้นแบบสุ่ม ต้องเทรนเต็มทั้งหัว
))
rm.print_trainable_parameters()
cast_trainable_to_fp32(rm)


def rm_scores(texts):
    """คะแนนสเกลาร์ r_phi ของแต่ละข้อความ (right padding)

    backbone เป็น fp16 แต่ score head (modules_to_save) ถูก cast เป็น fp32
    ต่างจาก causal LM ตรงที่ SeqCls มี matmul ที่หัวโดยตรง จึง dtype ชนกัน
    ("expected mat1 and mat2 to have the same dtype") ต้องห่อด้วย autocast
    ให้ backbone ทำงาน fp16 แล้วปล่อยหัว fp32 ทำต่อได้
    """
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=RM_MAX_LEN).to(DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
        logits = rm(**enc).logits
    return logits.squeeze(-1).float()


rm_trainables = [p for p in rm.parameters() if p.requires_grad]
opt_rm = torch.optim.AdamW(rm_trainables, lr=RM_LR)
try:
    scaler_rm = torch.amp.GradScaler("cuda", enabled=USE_FP16)
except (AttributeError, TypeError):
    scaler_rm = torch.cuda.amp.GradScaler(enabled=USE_FP16)

t0 = time.time()
RM_LOSS_LOG = []
rm.train()
for ep in range(RM_EPOCHS):
    order = list(range(len(RM_TRAIN)))
    random.Random(SEED + ep).shuffle(order)
    ep_losses, ep_correct = [], 0
    for i in range(0, len(order), RM_BATCH):
        batch = [RM_TRAIN[j] for j in order[i:i + RM_BATCH]]
        s_w = rm_scores([b["chosen"] for b in batch])     # r_phi(x, y_w)
        s_l = rm_scores([b["rejected"] for b in batch])   # r_phi(x, y_l)
        loss = -F.logsigmoid(s_w - s_l).mean()            # สมการ 3.1 ตรงตัว

        opt_rm.zero_grad(set_to_none=True)
        scaler_rm.scale(loss).backward()
        scaler_rm.unscale_(opt_rm)
        torch.nn.utils.clip_grad_norm_(rm_trainables, 1.0)
        scaler_rm.step(opt_rm)
        scaler_rm.update()

        ep_losses.append(float(loss))
        ep_correct += int((s_w > s_l).sum())
    mean_loss = sum(ep_losses) / len(ep_losses)
    RM_LOSS_LOG.append(mean_loss)
    print(f"epoch {ep + 1}/{RM_EPOCHS}  loss={mean_loss:.4f}  "
          f"train acc={ep_correct / len(RM_TRAIN):.2f}  "
          f"(loss เริ่มควรใกล้ ln 2 = {math.log(2):.3f} เพราะหัวคะแนนยังสุ่มอยู่)")

RM_TRAIN_TIME = time.time() - t0
print(f"\nเทรน Stage A เสร็จใน {RM_TRAIN_TIME / 60:.1f} นาที")

# --- คำพิพากษา: pairwise ranking accuracy บน 20 คู่ held-out -------------------
rm.eval()
margins = []
with torch.no_grad():
    for i in range(0, len(RM_HELDOUT), 4):
        batch = RM_HELDOUT[i:i + 4]
        s_w = rm_scores([b["chosen"] for b in batch])
        s_l = rm_scores([b["rejected"] for b in batch])
        margins.extend([float(v) for v in (s_w - s_l)])

n_correct = sum(1 for m in margins if m > 0)
n_total = len(margins)
rm_lo, rm_hi = wilson_ci(n_correct, n_total)

RM_STATS = {
    "n_train_pairs": len(RM_TRAIN),
    "n_heldout_pairs": n_total,
    "n_correct": n_correct,
    "accuracy": n_correct / n_total,
    "ci_low": rm_lo,
    "ci_high": rm_hi,
    "mean_margin": sum(margins) / n_total,
    "loss_per_epoch": RM_LOSS_LOG,
    "train_time_s": RM_TRAIN_TIME,
}

print(f"held-out pairwise accuracy = {100 * RM_STATS['accuracy']:.0f}% "
      f"({n_correct}/{n_total})  [Wilson 95% CI {rm_lo:.3f}-{rm_hi:.3f}]")
print(f"mean margin = {RM_STATS['mean_margin']:+.3f}")
if rm_lo > 0.5:
    print("=> CI ไม่คร่อม 0.5: RM ตัวนี้ดีกว่าโยนเหรียญอย่างมีนัยสำคัญ -- Stage A ผ่าน")
    print("   (คุณเพิ่งเทรน reward model ตัวแรกสำเร็จ ด้วยข้อมูลแค่ 80 แถว)")
else:
    print("=> CI ยังคร่อม 0.5: บน 20 คู่ เรายังแยก RM นี้จากการโยนเหรียญไม่ได้")
    print("   ต้องถูกอย่างน้อย 15/20 ถึงจะพ้น 0.5 -- นี่คือความจริงของ n เล็ก")
    print("   ไม่ใช่ความล้มเหลวของสมการ 3.1 (ลองเพิ่ม RM_EPOCHS หรือหาคู่เพิ่ม)")

# RM ทำหน้าที่ในบทเรียนครบแล้ว -- Stage B ใช้ rule reward (เหตุผลอยู่ในหัวข้อ 6)
# คืน VRAM ให้ PPO ซึ่งต้องถือ policy + ref + value head
del rm, opt_rm, scaler_rm, rm_trainables
gc.collect()
torch.cuda.empty_cache()
print("\nคืน VRAM ของ reward model แล้ว -- Stage B ใช้ rule reward เป็น stand-in")


### 7.2 Stage B — PPO เขียนเองจากศูนย์ ~120 บรรทัด

เราจะ **ไม่ใช้ `PPOTrainer` ของ TRL** และนี่คือการตัดสินใจที่ตั้งใจ ไม่ใช่ความดื้อ:
TRL ย้าย `PPOTrainer` ไปอยู่ `trl.experimental` แล้ว และประกาศแผนถอดออกใน 0.29.0 —
โค้ดที่สอนด้วยไลบรารีตัวนี้จะรันไม่ได้ในอีกไม่กี่เดือน ส่วนลูป PPO ที่เขียนเอง
ราว 120 บรรทัดจะรันได้ตราบเท่าที่ PyTorch ยังอยู่ และสำคัญกว่านั้น:
เขียนเองแล้วคุณจะ **รู้** ว่าทุกบรรทัดทำอะไร แบบเดียวกับ DPO loss 25 บรรทัดในตอนที่ 4

ชิ้นส่วนบนกระดาน:

| บทบาท | ของจริงในโน้ตบุ๊กนี้ | ต้นทุน VRAM เพิ่ม |
|---|---|---|
| policy $\pi_\theta$ | base + LoRA adapter (จาก Cell 1) | มีอยู่แล้ว |
| reference $\pi_{\text{ref}}$ | base ตัวเดิม **ปิด adapter** | **ศูนย์ไบต์** |
| reward $r_\phi$ | `rule_reward` (stand-in ของ RM — หัวข้อ 6) | ศูนย์ |
| value $V_\psi$ | MLP สองชั้นบน hidden state สุดท้าย | ~1M พารามิเตอร์ |

> **สามบรรทัดที่ถ้าพลาด การทดลองทั้งหมดเป็นโมฆะ**
>
> **1. `.clamp(-10, 10)` ก่อน `.exp()`** — ใน fp16 ค่า `exp(12)` = 162,754
> เกินเพดาน fp16 (65,504) ผลคือ `inf` แล้วกลายเป็น `NaN` ระบาดทั้ง batch ภายใน step เดียว
>
> **2. `logp_old` ต้องคำนวณครั้งเดียวตอน rollout แล้ว detach เก็บไว้** —
> ห้ามคำนวณใหม่ในลูป epoch ถ้าคำนวณใหม่ $\rho_t = 1$ เสมอ clip ไม่เคยทำงาน
> และ PPO ของคุณเสื่อมเป็น REINFORCE เงียบ ๆ โดยไม่มี error ใด ๆ
>
> **3. standardize คะแนนก่อนใช้** — สมการ 3.1 บอกแล้วว่าสเกลของ reward ไม่ถูกนิยาม
> rule reward ของเราอยู่ในช่วง 0 ถึง 1.2 ก็จริง แต่นิสัยนี้ต้องติดตัวไป
> ตอนใช้ RM จริงที่สเกลมั่วได้ตามใจ

รายละเอียดที่จงใจเลือก และพิมพ์ไว้ในโค้ดด้วย:

- **policy อยู่ใน `eval()` ตลอด** — dropout ปิด แต่ gradient ยังไหลปกติ
  เหตุผล: ถ้า dropout เปิด forward ตอน update จะไม่ตรงกับตอน rollout
  แล้ว $\rho_t \neq 1$ ตั้งแต่ epoch แรกทั้งที่ยังไม่ได้อัปเดตอะไรเลย
- **hidden state ที่ป้อน value head ถูก `.detach()`** — value loss มีสเกลใหญ่
  ถ้าปล่อยให้ไหลย้อนเข้า backbone มันจะเขียนทับความสามารถทางภาษา (กับดักข้อ 3 ของบทความ)
- **entropy ใช้ตัวประมาณจากตัวอย่างเดียว** $-\log\pi_\theta(a_t)$ ซึ่งมีค่าคาดหวัง
  เท่ากับ $\mathcal{H}$ พอดี — ประหยัด VRAM กว่าการกระจายเต็ม vocab 151,936 ช่อง
- **GAE ต่อ token, คะแนนงานบวกที่ token จริงตัวสุดท้าย** (ตำแหน่ง eos)
  ไม่ใช่ตำแหน่งสุดท้ายของ tensor ซึ่งอาจเป็น padding

ก่อนใช้จริง เซลล์ถัดไปตรวจคณิตศาสตร์ทุกชิ้นกับ **การคำนวณด้วยมือ**:
GAE บนลำดับ 3 ขั้นที่ไล่มือได้, clipped surrogate 4 กรณี, และการ standardize


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 (a) -- ชิ้นส่วนของ PPO: value head, hyperparameters, จุดย้อนกลับ
# -----------------------------------------------------------------------------
import copy

# --- โมเดลตัวที่สี่: value head = MLP สองชั้นบน hidden state สุดท้าย -------------
value_head = torch.nn.Sequential(
    torch.nn.Linear(cfg.hidden_size, cfg.hidden_size),
    torch.nn.Tanh(),
    torch.nn.Linear(cfg.hidden_size, 1),
).float().to(DEV)
torch.nn.init.zeros_(value_head[-1].weight)   # เริ่มที่ V=0 ทุก state -> advantage แรก ๆ
torch.nn.init.zeros_(value_head[-1].bias)     # อ่านตรงจาก reward ไม่ใช่จากหัวที่ยังสุ่ม

n_vh = sum(p.numel() for p in value_head.parameters())
print(f"value head: {n_vh / 1e6:.2f}M พารามิเตอร์ (จิ๋วเดียวเทียบกับ policy 596M)")

# --- hyperparameters ของ PPO (ตรงกับบทความหัวข้อ 7.2) --------------------------
EPS = 0.2            # ความกว้าง trust region
BETA_KL = 0.05       # ความตึงของสายจูง KL (รอบ ablation จะตั้งเป็น 0)
GAMMA = 1.0          # ธรรมเนียมงาน LLM
LAM = 0.95           # ปุ่ม bias-variance ของ GAE
C1, C2 = 0.5, 0.01   # น้ำหนัก value loss และ entropy bonus
PPO_EPOCHS = 4       # รีดค่า rollout เดิม 4 รอบ -- เหตุผลการมีอยู่ของ clip
ROLL_B = 4           # prompt ต่อ rollout (ลดจาก 8 -- ดูหมายเหตุ OOM)
# PPO ต้องถือ policy + reference + value head พร้อมกัน แล้ว generate ทั้ง batch
# รวดเดียวก่อน forward เข้า policy อีกรอบ วัดจริงบน T4 แล้ว ROLL_B=8 พยายามจอง
# 928 MiB ตอนเหลือ 575 MiB -> OOM. ลดเป็น 4 ยังผ่านโจทย์ครบเท่าเดิม (N_UPDATES เพิ่มเป็น 2 เท่า)
# แค่ทยอยทำทีละ 4 -- ผลการเทรนไม่เปลี่ยน
GEN_MAX_NEW = 128 if FAST_MODE else 200
MAX_PROMPT_TOK = 320
LR_POLICY = 1e-5
LR_VALUE = 1e-4
N_UPDATES = len(TRAIN_ITEMS) // ROLL_B   # 1 รอบผ่านโจทย์ครบทุกข้อ

policy_trainables = [p for p in policy.parameters() if p.requires_grad]
print(f"PPO: {N_UPDATES} updates x {PPO_EPOCHS} epochs | rollout ละ {ROLL_B} prompt "
      f"x {GEN_MAX_NEW} token")

# --- จุดย้อนกลับ: เก็บสภาพตั้งต้นไว้ เพื่อให้รอบ beta=0 เริ่มจากจุดเดียวกันเป๊ะ ----
INIT_POLICY_STATE = {n: p.detach().clone()
                     for n, p in policy.named_parameters() if p.requires_grad}
INIT_VALUE_STATE = copy.deepcopy(value_head.state_dict())


def snapshot_policy():
    return {n: p.detach().clone()
            for n, p in policy.named_parameters() if p.requires_grad}


def load_policy_state(state):
    with torch.no_grad():
        for n, p in policy.named_parameters():
            if n in state:
                p.copy_(state[n])


def restore_initial_state():
    """ย้อน policy + value head กลับสภาพตั้งต้น -- เงื่อนไขจำเป็นของ ablation ที่ยุติธรรม"""
    load_policy_state(INIT_POLICY_STATE)
    value_head.load_state_dict(INIT_VALUE_STATE)


def fresh_optim_and_scaler():
    opt_pi = torch.optim.AdamW(policy_trainables, lr=LR_POLICY)
    opt_v = torch.optim.AdamW(value_head.parameters(), lr=LR_VALUE)
    try:
        sc = torch.amp.GradScaler("cuda", enabled=USE_FP16)
    except (AttributeError, TypeError):
        sc = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    return opt_pi, opt_v, sc


VRAM_NOW = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\nVRAM ตอนนี้: {VRAM_NOW:.3f} GB")
print("โมเดลสี่บทบาทบนการ์ดใบเดียว: policy=base+adapter | ref=ปิด adapter (0 ไบต์)")
print("| reward=rule (0 ไบต์) | value head 1M -- นี่คือเหตุผลที่ตอนนี้รันได้บน T4 ฟรี")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 (b) -- แกนของ PPO ~120 บรรทัด + ตรวจคณิตศาสตร์กับมือก่อนใช้จริง
# -----------------------------------------------------------------------------
from types import SimpleNamespace


def gather_logprobs(logits, ids, prompt_len):
    """log pi(token) เฉพาะฝั่ง response: ตำแหน่ง P-1+t ของ logits ทำนาย token ที่ P+t

    ใช้ F.cross_entropy แบบ fused แทน log_softmax เต็ม [B,T,V] -- ประหยัด VRAM
    หลาย GB เมื่อ vocab ใหญ่ 151,936 ช่อง
    """
    sl = logits[:, prompt_len - 1:-1, :].float()
    targets = ids[:, prompt_len:]
    B, T, V = sl.shape
    nll = F.cross_entropy(sl.reshape(B * T, V), targets.reshape(B * T), reduction="none")
    return -nll.view(B, T)


def compute_gae(rewards, values, gamma=1.0, lam=0.95):
    """GAE (สมการ 3.4) -- rewards: [T] ของหนึ่ง response, values: [T+1] (ช่องสุดท้าย = 0)"""
    adv, acc = torch.zeros_like(rewards), 0.0
    for t in reversed(range(rewards.shape[0])):
        delta = rewards[t] + gamma * values[t + 1] - values[t]   # สมการ 3.4 บรรทัดบน
        acc = delta + gamma * lam * acc                          # A_t = delta_t + (gamma*lam) A_{t+1}
        adv[t] = acc
    return adv


def masked_mean(x, mask):
    return (x * mask).sum() / mask.sum().clamp(min=1)


@torch.no_grad()
def make_rollout(items, seed):
    """สุ่มคำตอบหนึ่งชุด แล้วแช่แข็งทุกอย่างที่ตอน update ห้ามคำนวณใหม่"""
    questions = [it["question"] for it in items]
    golds = [it["gold"] for it in items]

    tokenizer.padding_side = "left"      # decoder-only ต้อง pad ซ้ายตอน generate
    enc = tokenizer([to_gen_prompt(q) for q in questions], return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_PROMPT_TOK).to(DEV)
    tokenizer.padding_side = "right"

    torch.manual_seed(seed)
    out = policy.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW,
        do_sample=True,                  # RL ต้องสำรวจ -- นี่คือจุดที่ต่างจาก eval contract
        top_p=0.9,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    P = enc["input_ids"].shape[1]
    resp = out[:, P:]
    T = resp.shape[1]

    # mask ฝั่ง response: 1 จนถึง token หยุดตัวแรก (รวมตัวมันเอง), 0 หลังจากนั้น
    # Qwen3 หยุดได้ทั้งที่ <|im_end|> (eos) และ <|endoftext|> -- ต้องมองหาทั้งคู่
    is_stop = resp.eq(tokenizer.eos_token_id) | resp.eq(tokenizer.pad_token_id)
    has_stop = is_stop.any(-1)
    first_stop = torch.where(has_stop, is_stop.float().argmax(-1).long(),
                             torch.full((resp.shape[0],), T - 1, device=resp.device))
    ar = torch.arange(T, device=resp.device).unsqueeze(0)
    mask = (ar <= first_stop.unsqueeze(1)).float()

    attn = torch.cat([enc["attention_mask"], mask.long()], dim=1)
    pos = (attn.cumsum(-1) - 1).clamp(min=0)   # position ids ที่ถูกต้องเมื่อมี left padding

    texts = tokenizer.batch_decode(resp, skip_special_tokens=True)
    scores = torch.tensor([rule_reward(t, g) for t, g in zip(texts, golds)],
                          device=DEV, dtype=torch.float32)
    # standardize ภายใน batch (สมการ 3.1: สเกลของ reward ไม่ถูกนิยาม)
    scores_std = (scores - scores.mean()) / (scores.std() + 1e-8)

    # logp_old + logp_ref: คำนวณครั้งเดียว ที่นี่ แล้วแช่แข็ง (บรรทัดอันตรายข้อ 2)
    logp_old = gather_logprobs(
        policy(out, attention_mask=attn, position_ids=pos).logits, out, P)
    with policy.disable_adapter():       # reference = base ตัวเดิม ปิด adapter -- 0 ไบต์
        logp_ref = gather_logprobs(
            policy(out, attention_mask=attn, position_ids=pos).logits, out, P)

    return SimpleNamespace(
        ids=out, attn=attn, pos=pos, prompt_len=P, resp_mask=mask,
        logp_old=logp_old, logp_ref=logp_ref,
        scores=scores, scores_std=scores_std, texts=texts, golds=golds,
        kl_vs_ref=float(masked_mean(logp_old - logp_ref, mask)),
        mean_len=float(mask.sum(-1).mean()),
    )


def ppo_update(ro, eps=0.2, beta=0.05):
    """หนึ่งก้าวของสมการ 3.5 บน rollout ที่แช่แข็งแล้ว -- คืน loss กับสถิติ"""
    out = policy(ro.ids, attention_mask=ro.attn, position_ids=ro.pos,
                 output_hidden_states=True)                    # forward เดียวได้สองอย่าง
    logp = gather_logprobs(out.logits, ro.ids, ro.prompt_len)  # [B, T] มี gradient
    with torch.no_grad(), policy.disable_adapter():
        logp_ref = gather_logprobs(
            policy(ro.ids, attention_mask=ro.attn, position_ids=ro.pos).logits,
            ro.ids, ro.prompt_len)

    m = ro.resp_mask
    # reward ต่อ token = ค่าปรับ KL ทุกตำแหน่ง + คะแนนงานที่ token จริงตัวสุดท้าย (สมการ 3.2)
    rew = -beta * (logp.detach() - logp_ref) * m
    last = (m.sum(-1) - 1).long()
    rew[torch.arange(rew.shape[0], device=DEV), last] += ro.scores_std

    hidden = out.hidden_states[-1].detach().float()   # detach = ตัด gradient เข้าลำต้น (กับดักข้อ 3)
    hidden = hidden[:, ro.prompt_len - 1:-1, :]       # state "ก่อนปล่อย" token ที่ t
    values = value_head(hidden).squeeze(-1) * m

    adv_raw = torch.stack([
        compute_gae(r, F.pad(v, (0, 1)), GAMMA, LAM)  # ช่องสุดท้าย = 0: หลัง eos ไม่มีอนาคต
        for r, v in zip(rew, values)
    ]) * m
    returns = (adv_raw + values).detach()             # เป้าของ value head ใช้สเกลจริง
    flat = adv_raw[m.bool()]
    adv = (adv_raw - flat.mean()) / (flat.std() + 1e-8)   # ส่วนที่เข้า surrogate ใช้สเกลมาตรฐาน

    log_ratio = (logp - ro.logp_old).clamp(-10, 10)   # กัน fp16 overflow (บรรทัดอันตรายข้อ 1)
    ratio = log_ratio.exp()                           # rho_t (สมการ 3.3)
    adv_d = adv.detach()
    surr = torch.min(ratio * adv_d, ratio.clamp(1 - eps, 1 + eps) * adv_d)

    policy_loss = -masked_mean(surr, m)
    value_loss = masked_mean((values - returns).pow(2), m)
    entropy_est = -masked_mean(logp, m)   # E[-log pi] = H : ตัวประมาณจากตัวอย่างเดียว
    loss = policy_loss + C1 * value_loss - C2 * entropy_est   # สมการ 3.5

    stats = {
        "policy_loss": float(policy_loss),
        "value_loss": float(value_loss),
        "entropy": float(entropy_est),
        "mean_ratio": float(masked_mean(ratio, m)),
        "clip_frac": float(masked_mean(((ratio - 1).abs() > eps).float(), m)),
    }
    return loss, stats


# =============================================================================
# ตรวจคณิตศาสตร์กับมือ ก่อนปล่อยให้มันเทรนจริง (รันบน CPU ก็ได้ ไม่ใช้โมเดล)
# =============================================================================
# --- (1) GAE บนลำดับ 3 ขั้นที่ไล่มือได้ ---------------------------------------
_r = torch.tensor([1.0, 0.0, 0.5])
_v = torch.tensor([0.2, 0.1, 0.3, 0.0])
_adv = compute_gae(_r, _v, gamma=1.0, lam=0.95)
# ไล่มือ: d2 = 0.5+0-0.3 = 0.2        -> A2 = 0.2
#         d1 = 0.0+0.3-0.1 = 0.2      -> A1 = 0.2 + 0.95*0.2  = 0.39
#         d0 = 1.0+0.1-0.2 = 0.9      -> A0 = 0.9 + 0.95*0.39 = 1.2705
_hand = torch.tensor([1.2705, 0.39, 0.2])
assert torch.allclose(_adv, _hand, atol=1e-6), f"GAE ไม่ตรงกับมือ: {_adv}"
print(f"GAE self-check   : code={[round(float(x), 4) for x in _adv]} == มือ {_hand.tolist()} -> ผ่าน")

# lambda=0 ต้องยุบเป็น TD(0) และ lambda=1 ต้องยุบเป็น (ผลตอบแทนจริง - V)
assert torch.allclose(compute_gae(_r, _v, 1.0, 0.0), torch.tensor([0.9, 0.2, 0.2]), atol=1e-6)
assert torch.allclose(compute_gae(_r, _v, 1.0, 1.0),
                      torch.tensor([1.5, 0.5, 0.5]) - _v[:3], atol=1e-6)
print("GAE ขอบเขต       : lambda=0 -> TD(0), lambda=1 -> MC - V -> ผ่าน")

# --- (2) clipped surrogate 4 กรณี เทียบกับมือ (eps=0.2) -------------------------
_ratio = torch.tensor([0.5, 1.5, 0.5, 1.5])
_advs = torch.tensor([1.0, 1.0, -1.0, -1.0])
_surr = torch.min(_ratio * _advs, _ratio.clamp(0.8, 1.2) * _advs)
# มือ: (0.5,+1)->0.5 | (1.5,+1)->1.2 เพดาน | (0.5,-1)->-0.8 เลือกแย่กว่า | (1.5,-1)->-1.5
assert torch.allclose(_surr, torch.tensor([0.5, 1.2, -0.8, -1.5])), f"clip ไม่ตรงกับมือ: {_surr}"
print(f"clip self-check  : {[round(float(x), 2) for x in _surr]} == มือ [0.5, 1.2, -0.8, -1.5] -> ผ่าน")

# --- (3) ทำไมต้อง clamp: exp(12) ระเบิดใน fp16 --------------------------------
assert torch.isinf(torch.tensor([12.0]).exp().half()).all()
assert torch.isfinite(torch.tensor([12.0]).clamp(-10, 10).exp().half()).all()
print("clamp self-check : exp(12) ใน fp16 = inf แต่ exp(clamp(12)) จำกัดอยู่ -> ผ่าน")

# --- (4) standardize: mean 0, std 1 และกรณีคะแนนเท่ากันหมด = ไม่มีสัญญาณ --------
_s = torch.tensor([1.2, 0.2, 1.0, 0.2])
_std = (_s - _s.mean()) / (_s.std() + 1e-8)
assert abs(float(_std.mean())) < 1e-6 and abs(float(_std.std()) - 1.0) < 1e-4
_tied = torch.ones(4)
_tied_std = (_tied - _tied.mean()) / (_tied.std() + 1e-8)
assert torch.allclose(_tied_std, torch.zeros(4))
print("standardize      : mean=0, std=1 | คะแนนเท่ากันหมด -> ศูนย์ล้วน (ไม่มีสัญญาณ) -> ผ่าน")
print()
print("คณิตศาสตร์ทุกชิ้นตรงกับมือ -- ตอนนี้ถึงจะมีสิทธิ์กดเทรน")


---

## 8. ผลลัพธ์ (Results) — รัน PPO ใต้สายจูง KL

รันแรก: $\beta = 0.05$ — ทุก update จะพิมพ์ **สามตัวเลขพร้อมกัน** และห้ามอ่านแยกกัน:

1. **reward เฉลี่ยต่อ rollout** (สเกลดิบ 0-1.2) — ควรไต่ขึ้น: นี่คือสิ่งที่เราซื้อ
2. **KL ต่อ token เทียบ $\pi_{\text{ref}}$** — ควรโตแล้ว **อิ่มตัว** ใต้เพดานที่ $\beta$
   กำหนด: นี่คือราคาที่จ่าย
3. **ความยาวคำตอบเฉลี่ย** — ตัวจับโรค: ความยาวพุ่งหรือดิ่งผิดปกติคือสัญญาณแรกของ
   policy ที่กำลังเพี้ยน

> **ห้ามอ่านเส้น reward โดยไม่มีเส้น KL ประกบ — เด็ดขาด**
> reward ขึ้น + KL อิ่มตัว = กำลังเรียนรู้ภายใต้สายจูง
> reward ขึ้น + KL พุ่งไม่หยุด = กำลังหนีออกจากภาษา ไปหาช่องโหว่ของกรรมการ
> เส้นเดียวกันคนละบริบท ความหมายตรงข้ามกันเป๊ะ

สองหมายเหตุที่จะช่วยตอนอ่าน log จริง:

- ที่ epoch แรกของ update แรก `ratio` เฉลี่ยต้องเท่ากับ **1.0 เป๊ะ** เพราะ policy
  ยังไม่ขยับจาก snapshot — โค้ด `assert` ข้อนี้ไว้จริง ถ้าไม่ผ่านแปลว่า `logp_old`
  รั่ว (บรรทัดอันตรายข้อ 2)
- ถ้า batch ไหนโมเดลตอบผิดหมดทุกข้อ (หรือถูกหมด) reward จะเท่ากันทั้ง batch
  แล้ว advantage หลัง standardize คือศูนย์ล้วน — **ไม่มีสัญญาณให้เรียน**
  โน้ตบุ๊กพิมพ์เตือนเมื่อเจอ batch แบบนี้ นี่ไม่ใช่ bug แต่เป็นธรรมชาติของ RL
  บนโมเดลที่ยังอ่อนกับงาน (ตอนที่ 5 จะเจอปัญหาเดียวกันในบริบท GRPO)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8 -- probe ก่อนเทรน + ลูปเทรน PPO + รันจริงรอบที่ 1 (beta = 0.05)
# -----------------------------------------------------------------------------
@torch.no_grad()
def generate_probe(items, max_new_tokens=None):
    """greedy generation บนโจทย์ held-out + KL ต่อ token ของแต่ละคำตอบเทียบ ref"""
    max_new_tokens = max_new_tokens or GEN_MAX_NEW
    questions = [it["question"] for it in items]
    tokenizer.padding_side = "left"
    enc = tokenizer([to_gen_prompt(q) for q in questions], return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_PROMPT_TOK).to(DEV)
    tokenizer.padding_side = "right"
    torch.manual_seed(SEED)
    out = policy.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.pad_token_id)
    P = enc["input_ids"].shape[1]
    resp = out[:, P:]
    T = resp.shape[1]
    is_stop = resp.eq(tokenizer.eos_token_id) | resp.eq(tokenizer.pad_token_id)
    has_stop = is_stop.any(-1)
    first_stop = torch.where(has_stop, is_stop.float().argmax(-1).long(),
                             torch.full((resp.shape[0],), T - 1, device=resp.device))
    mask = (torch.arange(T, device=resp.device).unsqueeze(0) <= first_stop.unsqueeze(1)).float()
    attn = torch.cat([enc["attention_mask"], mask.long()], dim=1)
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    lp = gather_logprobs(policy(out, attention_mask=attn, position_ids=pos).logits, out, P)
    with policy.disable_adapter():
        lp_ref = gather_logprobs(policy(out, attention_mask=attn, position_ids=pos).logits, out, P)
    kl_per_seq = ((lp - lp_ref) * mask).sum(-1) / mask.sum(-1).clamp(min=1)
    texts = tokenizer.batch_decode(resp, skip_special_tokens=True)
    return [{
        "text": t,
        "kl": float(k),
        "reward": rule_reward(t, it["gold"]),
        "th_ratio": th_ratio(t),
        "tokens": int(n),
    } for t, k, n, it in zip(texts, kl_per_seq, mask.sum(-1), items)]


def run_ppo(beta, tag):
    """เทรน PPO หนึ่งรอบเต็มจากสภาพตั้งต้นเดิมเป๊ะ -- คืน history ต่อ update"""
    restore_initial_state()                    # ทุกรอบเริ่มจากจุดเดียวกัน -- ablation ที่ยุติธรรม
    opt_pi, opt_v, sc = fresh_optim_and_scaler()
    policy.eval()                              # ปิด dropout -- gradient ยังไหลปกติ
    history = []
    t_start = time.time()
    print(f"=== PPO {tag}: beta = {beta} | {N_UPDATES} updates x {PPO_EPOCHS} epochs ===")
    print(f"{'upd':>4s} {'reward':>7s} {'KL/token':>9s} {'len':>6s} {'clip%':>6s} {'ratio':>6s}")

    for upd in range(N_UPDATES):
        items = TRAIN_ITEMS[upd * ROLL_B:(upd + 1) * ROLL_B]
        ro = make_rollout(items, seed=SEED + 1000 * upd)

        if float(ro.scores.std()) == 0.0:
            print(f"  [เตือน] update {upd}: reward เท่ากันทั้ง batch "
                  f"({float(ro.scores[0]):.1f}) -> advantage เป็นศูนย์ ไม่มีสัญญาณ")

        last_stats = {}
        for ep in range(PPO_EPOCHS):
            loss, stats = ppo_update(ro, eps=EPS, beta=beta)
            if upd == 0 and ep == 0:
                # policy ยังไม่ขยับจาก snapshot -> ratio ต้องเป็น 1 เป๊ะ
                assert abs(stats["mean_ratio"] - 1.0) < 1e-3, (
                    f"ratio epoch แรก = {stats['mean_ratio']} != 1 -- logp_old รั่ว (ข้อ 2)")
            opt_pi.zero_grad(set_to_none=True)
            opt_v.zero_grad(set_to_none=True)
            sc.scale(loss).backward()
            sc.unscale_(opt_pi)
            sc.unscale_(opt_v)
            torch.nn.utils.clip_grad_norm_(policy_trainables, 1.0)
            torch.nn.utils.clip_grad_norm_(value_head.parameters(), 1.0)
            sc.step(opt_pi)
            sc.step(opt_v)
            sc.update()
            last_stats = stats

        history.append({
            "update": upd,
            "reward_mean": float(ro.scores.mean()),
            "reward_std": float(ro.scores.std()),
            "kl_vs_ref": ro.kl_vs_ref,
            "resp_len": ro.mean_len,
            **last_stats,
        })
        h = history[-1]
        print(f"{upd:4d} {h['reward_mean']:7.3f} {h['kl_vs_ref']:9.4f} "
              f"{h['resp_len']:6.0f} {100 * h['clip_frac']:5.1f}% {h['mean_ratio']:6.3f}")

    elapsed = time.time() - t_start
    print(f"รอบ {tag} ใช้เวลา {elapsed / 60:.1f} นาที")
    return history, elapsed


# --- probe ตั้งต้น (ก่อน PPO ทุกชนิด) -------------------------------------------
policy.eval()
PROBE_BEFORE = generate_probe(PROBE_ITEMS)
print("ตัวอย่างคำตอบตั้งต้นบนโจทย์ held-out:")
print("  ", PROBE_BEFORE[0]["text"][:200].replace(chr(10), " "))
print(f"   reward={PROBE_BEFORE[0]['reward']:.1f}  th_ratio={PROBE_BEFORE[0]['th_ratio']:.2f}")

# --- รันจริงรอบที่ 1: มีสายจูง ---------------------------------------------------
torch.cuda.reset_peak_memory_stats()
HIST_KL, TIME_KL = run_ppo(BETA_KL, tag=f"beta={BETA_KL}")
VRAM_PEAK_PPO = torch.cuda.max_memory_allocated() / (1024 ** 3)
print(f"VRAM peak ระหว่าง PPO: {VRAM_PEAK_PPO:.2f} GB")

PROBE_KL = generate_probe(PROBE_ITEMS)
STATE_AFTER_KL = snapshot_policy()   # เก็บ adapter หลังรอบ KL ไว้ -- ใช้วัด KobEval ในหัวข้อ 9

reward_first, reward_last = HIST_KL[0]["reward_mean"], HIST_KL[-1]["reward_mean"]
kl_last = HIST_KL[-1]["kl_vs_ref"]
print(f"\nสรุปรอบ beta={BETA_KL}: reward {reward_first:.3f} -> {reward_last:.3f} | "
      f"KL สุดท้าย {kl_last:.4f} ต่อ token")
print("ถ้า reward ขยับขึ้นโดย KL ไม่บาน = ลูปทำงานใต้สายจูงตามสมการ 3.2")


---

## 9. เปรียบเทียบ (Comparison) — ถอดสายจูง KL ออก

รอบที่สอง: $\beta = 0$ โดยแตะอย่างอื่นเลยแม้แต่ตัวเดียว — ข้อมูลเดิม seed เดิม
จำนวน step เดิม จุดตั้งต้นเดิม (โค้ด `restore_initial_state()` การันตีข้อนี้)
**ตัวแปรเดียวที่ต่างคือสายจูง**

รูปแบบที่ **คาดว่าจะเห็น**: reward ไต่เท่าเดิมหรือเร็วกว่า แต่ KL ทะยานไม่มีเพดาน
และคำตอบเริ่มเสื่อมสภาพ — วนซ้ำ สั้นผิดปกติ หรือกลายเป็นสูตรสำเร็จที่ยัดตัวเลข
ไว้ท้ายประโยค เพราะ `rule_reward` มองเห็นแค่เลขท้ายกับสัดส่วนอักขระไทย
**ทุกอย่างที่กรรมการมองไม่เห็นคือของฟรีที่โมเดลทิ้งได้** — นี่คือ Goodhart's law
ในเวอร์ชันที่วัดได้

ถ้าเห็นอย่างอื่น ให้ตีความแบบนี้ (อย่าเพิ่งสรุปว่าโค้ดพัง):

- **ทั้งสองรอบแทบไม่ต่างกัน** → advantage เป็นศูนย์เกือบตลอด เช็ค log ว่ามีคำเตือน
  "reward เท่ากันทั้ง batch" บ่อยแค่ไหน — โมเดล 0.6B ตอบโจทย์ GSM8K ผิดเป็นส่วนใหญ่
  ความหลากหลายของ reward จึงมาจากโบนัสภาษาไทยเป็นหลัก
- **β = 0 แล้ว KL ไม่ทะยาน** → งบ ~8 updates สั้นเกินกว่าที่การ hack จะสุกงอม —
  เพิ่มรอบ (วนซ้ำ `TRAIN_ITEMS` หลายรอบ) แล้วดูใหม่ อย่าเพิ่งสรุปว่า "ไม่มี hacking"
- **β = 0.05 แต่ KL ก็ยังทะยาน** → เกือบแน่นอนว่าลืม standardize คะแนน
  ทำให้สเกล reward ท่วม β หรือไม่ก็ LR สูงเกิน

ท้ายเซลล์ถัดไปคือ **hacking exhibits**: คำตอบจริงจากรัน $\beta = 0$
พร้อมค่า KL ของมันรายตัว — เราไม่แต่งตัวอย่าง degenerate ขึ้นเอง
ทั้งซีรีส์นี้ยืนอยู่บนกติกาว่าไม่มี output ที่ invent ขึ้นมา


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.1 -- ablation: beta = 0 (ไม่มีสายจูง) + hacking exhibits
# -----------------------------------------------------------------------------
HIST_NOKL, TIME_NOKL = run_ppo(0.0, tag="beta=0 (ablation)")
PROBE_NOKL = generate_probe(PROBE_ITEMS)

print()
print("=== เทียบปลายทางของสองรอบ (ตัวเลขจาก log จริงด้านบน) ===")
print(f"{'':16s} {'reward สุดท้าย':>15s} {'KL สุดท้าย':>12s} {'len สุดท้าย':>12s}")
for tag, hist in [(f"beta={BETA_KL}", HIST_KL), ("beta=0", HIST_NOKL)]:
    h = hist[-1]
    print(f"{tag:16s} {h['reward_mean']:15.3f} {h['kl_vs_ref']:12.4f} {h['resp_len']:12.0f}")

kl_ratio = HIST_NOKL[-1]["kl_vs_ref"] / max(abs(HIST_KL[-1]["kl_vs_ref"]), 1e-6)
print(f"\nKL ปลายทางของ beta=0 เป็น {kl_ratio:.1f} เท่าของรอบมีสายจูง")
print("(ถ้าตัวคูณนี้ยังไม่ใหญ่ ให้เพิ่มจำนวนรอบ -- การ hack ต้องการเวลาสุกงอม)")

print()
print("=== hacking exhibits: คำตอบจริงจากรอบ beta=0 พร้อม KL รายตัว ===")
print("(อ่านหาอาการ: วนซ้ำ, ทิ้งภาษา, สูตรสำเร็จยัดตัวเลข -- เทียบกับคำตอบตั้งต้น)")
for i in range(min(3, len(PROBE_ITEMS))):
    q = PROBE_ITEMS[i]["question"]
    print(f"\n[{i + 1}] โจทย์  : {q[:110].replace(chr(10), ' ')}")
    print(f"    ตั้งต้น (KL={PROBE_BEFORE[i]['kl']:+.3f}, r={PROBE_BEFORE[i]['reward']:.1f}): "
          f"{PROBE_BEFORE[i]['text'][:170].replace(chr(10), ' ')}")
    print(f"    β=0.05 (KL={PROBE_KL[i]['kl']:+.3f}, r={PROBE_KL[i]['reward']:.1f}): "
          f"{PROBE_KL[i]['text'][:170].replace(chr(10), ' ')}")
    print(f"    β=0    (KL={PROBE_NOKL[i]['kl']:+.3f}, r={PROBE_NOKL[i]['reward']:.1f}): "
          f"{PROBE_NOKL[i]['text'][:170].replace(chr(10), ' ')}")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.2 -- triptych: reward / KL / ความยาว ของทั้งสองรอบ + export samples.json
# -----------------------------------------------------------------------------
use_thai_font()

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
panels = [
    ("reward_mean", "reward เฉลี่ยต่อ rollout (ดิบ 0-1.2)", "สิ่งที่เราซื้อ"),
    ("kl_vs_ref", "KL ต่อ token เทียบ reference", "ราคาที่จ่าย -- เส้นที่ต้องประกบอ่าน"),
    ("resp_len", "ความยาวคำตอบเฉลี่ย (token)", "ตัวจับโรคของ policy ที่กำลังเพี้ยน"),
]
for ax, (key, title, subtitle) in zip(axes, panels):
    for hist, color, label in [(HIST_KL, "#2563eb", f"β = {BETA_KL} (มีสายจูง)"),
                               (HIST_NOKL, "#dc2626", "β = 0 (ถอดสายจูง)")]:
        xs = [h["update"] for h in hist]
        ys = [h[key] for h in hist]
        ax.plot(xs, ys, "-o", ms=4, lw=2, color=color, label=label)
    ax.set_xlabel("update")
    ax.set_title(f"{title}\n{subtitle}", fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)
axes[0].legend(fontsize=9)
fig.suptitle("อ่านสามแผงพร้อมกันเสมอ: reward ที่ไม่มี KL ประกบ อ่านค่าอะไรไม่ได้เลย",
             fontsize=11)
fig.tight_layout()
fig.savefig("fig_ppo_triptych.png", dpi=150, bbox_inches="tight")
plt.show()
print("บันทึกรูปแล้ว: fig_ppo_triptych.png")

# --- export ตัวอย่าง generation ทั้งสามระบบลง samples.json ----------------------
# (วิดเจ็ต BeforeAfterCompare บนบล็อกอ่านไฟล์นี้ไปแสดงคำตอบจริง ไม่ใช่คำตอบที่แต่งขึ้น)
sample_items = []
for i, it in enumerate(PROBE_ITEMS):
    outputs = {
        "before": PROBE_BEFORE[i]["text"],
        "ppo_kl": PROBE_KL[i]["text"],
        "ppo_no_kl": PROBE_NOKL[i]["text"],
    }
    metrics = {}
    for key, probe in (("before", PROBE_BEFORE), ("ppo_kl", PROBE_KL), ("ppo_no_kl", PROBE_NOKL)):
        metrics[key] = {
            "tokens": probe[i]["tokens"],
            "thai_ratio": round(probe[i]["th_ratio"], 4),
            "reward": round(probe[i]["reward"], 2),
            "kl_per_token": round(probe[i]["kl"], 4),
        }
    sample_items.append({"prompt": it["question"], "outputs": outputs, "metrics": metrics})

with open("samples.json", "w", encoding="utf-8") as f:
    json.dump({"items": sample_items}, f, ensure_ascii=False, indent=2)
print(f"เขียน samples.json แล้ว ({len(sample_items)} โจทย์ x 3 ระบบ: "
      "before / ppo_kl / ppo_no_kl)")

mean_r = lambda probe: sum(p["reward"] for p in probe) / len(probe)
mean_kl = lambda probe: sum(p["kl"] for p in probe) / len(probe)
print()
print(f"{'ระบบ':14s} {'reward เฉลี่ย':>13s} {'KL เฉลี่ย':>10s}")
print("-" * 40)
print(f"{'ตั้งต้น':14s} {mean_r(PROBE_BEFORE):13.2f} {mean_kl(PROBE_BEFORE):10.3f}")
print(f"{'β=' + str(BETA_KL):14s} {mean_r(PROBE_KL):13.2f} {mean_kl(PROBE_KL):10.3f}")
print(f"{'β=0':14s} {mean_r(PROBE_NOKL):13.2f} {mean_kl(PROBE_NOKL):10.3f}")
print()
print("คะแนนที่โกงมาไม่ควรโอนย้ายไปโจทย์ใหม่ -- ถ้าแถว β=0 ได้ reward บน probe")
print("สูงจริง ให้ไปอ่านตัวคำตอบใน samples.json ว่ามันยังเป็นภาษาอยู่หรือเปล่า")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.3 -- วัด KobEval-TH ซ้ำด้วยสัญญาเดิม (นโยบายรอบ beta=0.05 -- ตัวที่แนะนำ)
# -----------------------------------------------------------------------------
# ตอนนี้น้ำหนักใน policy คือผลของรอบ beta=0 (รันทีหลัง) -- ย้อนกลับไปรอบมีสายจูงก่อน
load_policy_state(STATE_AFTER_KL)
policy.eval()

t0 = time.time()
after = evaluate(
    policy,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name=f"policy + PPO (β={BETA_KL})",
    out_path="results_after.json",
    extra={"train_time_s": TIME_KL, "vram_peak_gb": VRAM_PEAK_PPO},
)
print(f"ใช้เวลาวัดผลหลัง PPO: {time.time() - t0:.0f} วินาที\n")

table = compare(baseline, after, out_path="results_table.json")

plot_before_after(
    baseline,
    after,
    slices=KOBEVAL_SLICES,
    title=f"ตอนที่ 3: ก่อน vs หลัง PPO (β={BETA_KL})",
    out_path="fig_before_after.png",
    results_json="results_before_after.json",
)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.4 -- รวมทุกตัวเลขลง results.json ไฟล์เดียว เพื่อให้วิดเจ็ตบนบล็อกอ่านค่าจริง
# -----------------------------------------------------------------------------
from kobeval import write_results

payload = {
    "post": 3,
    "slug": "llm-03-rlhf-ppo",
    "notebook": "03_rlhf_ppo.ipynb",
    "model": MODEL_ID,
    "policy_source": POLICY_SOURCE,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "datasets": ["iapp/dpo_thai_tutorial", "VISAI-AI/gsm8k-thai"],
    "reward_model": RM_STATS,
    "rule_reward": {
        "correct_final_int": 1.0,
        "thai_bonus": 0.2,
        "thai_bonus_threshold": 0.5,
        "thai_digits_normalised": True,
    },
    "hyperparameters": {
        "eps": EPS,
        "beta_kl": BETA_KL,
        "gamma": GAMMA,
        "lam": LAM,
        "c1": C1,
        "c2": C2,
        "ppo_epochs": PPO_EPOCHS,
        "n_updates": N_UPDATES,
        "rollout_batch": ROLL_B,
        "gen_max_new_tokens": GEN_MAX_NEW,
        "lr_policy": LR_POLICY,
        "lr_value": LR_VALUE,
        "value_head_params": n_vh,
        "fast_mode": FAST_MODE,
    },
    "training": {
        "beta_kl_run": {"history": HIST_KL, "train_time_s": TIME_KL},
        "beta_zero_run": {"history": HIST_NOKL, "train_time_s": TIME_NOKL},
        "vram_peak_gb": VRAM_PEAK_PPO,
    },
    "probe": {
        "n": len(PROBE_ITEMS),
        "before": {"mean_reward": mean_r(PROBE_BEFORE), "mean_kl": mean_kl(PROBE_BEFORE)},
        "ppo_kl": {"mean_reward": mean_r(PROBE_KL), "mean_kl": mean_kl(PROBE_KL)},
        "ppo_no_kl": {"mean_reward": mean_r(PROBE_NOKL), "mean_kl": mean_kl(PROBE_NOKL)},
    },
    "kobeval": {
        "slices_run": KOBEVAL_SLICES,
        "before": {
            "model": baseline["model"],
            "accuracy": {k: v["accuracy"] for k, v in baseline["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in baseline["slices"].items()},
            "th_ratio": baseline["overall"]["th_ratio"],
        },
        "after": {
            "model": after["model"],
            "accuracy": {k: v["accuracy"] for k, v in after["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in after["slices"].items()},
            "th_ratio": after["overall"]["th_ratio"],
        },
        "mcnemar": table.get("mcnemar"),
        "markdown_table": table["markdown"],
    },
    "samples_file": "samples.json",
    "figures": [
        "fig_ppo_clip_gae.png",
        "fig_ppo_triptych.png",
        "fig_before_after.png",
    ],
}

path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}")
print(json.dumps({
    "reward_model": {k: payload["reward_model"][k]
                     for k in ("accuracy", "ci_low", "ci_high", "n_heldout_pairs")},
    "probe": payload["probe"],
}, ensure_ascii=False, indent=2))


---

## 10. สรุป (Summary)

- **RLHF = เดินอ้อมสองขั้น** เพื่อ optimize สิ่งที่หาอนุพันธ์ไม่ได้: เทรนกรรมการ
  ($r_\phi$ — Stage A ของเราผ่านการวัดด้วย Wilson CI ไม่ใช่ความรู้สึก)
  แล้วใช้ RL วิ่งเข้าหาคะแนนของกรรมการ (Stage B)
- **Bradley–Terry เห็นแค่ผลต่าง** — สเกลสัมบูรณ์ของ reward ไม่ถูกนิยาม
  จึงต้อง standardize ก่อนใช้เสมอ และเราตรวจพฤติกรรมนี้ด้วย assert แล้วจริง
- **สมการแม่ $\max\ \mathbb{E}[r] - \beta\,\mathbb{D}_{\text{KL}}$ คือสมการเดียวที่ต้องท่องจำ** —
  ตอนที่ 4 (DPO) และตอนที่ 5 (GRPO) คือการแก้สมการนี้ด้วยวิธีอื่น
- **KL ไม่ใช่ regularizer** — มันคือเงื่อนไขหยุดเพียงอย่างเดียวของระบบ
  รอบ $\beta = 0$ ของเราคือหลักฐานที่วัดได้ ไม่ใช่นิทานสอนใจ
- **min + clip = มองโลกแง่ร้ายโดยดีไซน์**: ได้จำกัด เสียไม่จำกัด — และเราตรวจ
  ทั้ง 4 กรณีกับการคำนวณด้วยมือแล้ว
- **GAE คือปุ่ม bias–variance** และ $V_\psi$ ที่มันพึ่งพาคือโมเดลตัวที่สี่
  ที่ GRPO จะลบทิ้งในตอนที่ 5
- **สองบั๊กเงียบที่สุดของ PPO**: คำนวณ `logp_old` ใหม่ในลูป (ratio = 1 ตลอดกาล
  — โค้ดเรา assert กันไว้) และลืม `.clamp` ก่อน `.exp()` ใน fp16 (NaN ทั้ง batch)
- **PPO แพงเพราะโครงสร้าง ไม่ใช่เพราะเขียนโค้ดห่วย**: โมเดล 4 บทบาท +
  hyperparameter ร่วมสิบตัวคือราคาหน้าตั๋ว — LoRA จากตอนที่ 2 คือเหตุผลเดียว
  ที่ทั้งหมดนี้ยัดลง T4 ฟรีได้

**ตอนต่อไป:** DPO — ลบทั้ง reward model และ RL loop ทิ้ง **ด้วยพีชคณิตล้วน ๆ**
จากสมการแม่ 3.2 ที่คุณเพิ่งท่องจำไป แล้วพิสูจน์ด้วย `assert` ว่า loss ที่เขียนเอง
25 บรรทัดตรงกับไลบรารีที่คนทั้งโลกใช้


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **64 prompt กับ rule-based reward คือการสาธิตอัลกอริทึม ไม่ใช่การทำ RLHF** —
   RLHF จริงใช้ RM ระดับ 7B ขึ้นไปที่เทรนจาก preference ของมนุษย์หลักหมื่นถึง
   หลักล้านคู่ และแยก rollout fleet ออกจากเครื่องเทรน สิ่งที่พิสูจน์ได้จริงมีสองอย่าง:
   **ลูป PPO ที่เขียนเองทำงานถูกต้อง** และ **กลไกของ reward hacking มีจริง**
   อย่าเอาผลนี้ไปอ้างว่าได้โมเดลไทยที่ align แล้ว

2. **Stage B ไม่ได้ใช้ reward model จาก Stage A** — RM ที่เทรนจาก 100 คู่อ่อนเกินกว่า
   จะรับแรงกดดันของ PPO ถ้าใช้มัน เราจะแยกไม่ออกว่าลูปผิดหรือ RM อ่อน
   นี่คือการตัดตัวแปรอย่างตั้งใจ แต่ก็แปลว่าโน้ตบุ๊กนี้ **ไม่ได้** สาธิตการต่อท่อ
   A → B แบบระบบจริง

3. **reward model วัดบน held-out แค่ 20 คู่** — Wilson CI จึงกว้างมาก ต้องถูก
   อย่างน้อย 15/20 ถึงจะอ้างว่าดีกว่าโยนเหรียญ ถ้ารอบของคุณไม่ถึง นั่นคือข้อมูล
   ไม่ใช่ข้อแก้ตัว

4. **โมเดลเล็กตอบโจทย์ GSM8K ผิดเป็นส่วนใหญ่** — reward ในหลาย batch จึงแทบ
   ไม่มีความหลากหลาย advantage หลัง standardize ใกล้ศูนย์ และสัญญาณการเรียนมาจาก
   โบนัสภาษาไทยเป็นหลัก ผล accuracy บน TH-MATH จึงอาจไม่ขยับอย่างมีนัยสำคัญ
   (ดูค่า p จาก McNemar ที่ `compare()` พิมพ์เสมอ)

5. **งบ ~8 updates ต่อรอบสั้นมาก** — reward hacking ที่สุกงอมต้องการเวลามากกว่านี้
   ถ้ารอบ $\beta = 0$ ของคุณยังไม่เห็น KL ทะยานชัด ให้เพิ่มจำนวนรอบ
   อย่าเพิ่งสรุปว่าสายจูงไม่จำเป็น

6. **entropy ใช้ตัวประมาณจากตัวอย่างเดียว** และ KL รายงานเป็นค่าประมาณ
   Monte-Carlo บนเส้นทางที่สุ่มได้ (อาจติดลบได้เมื่อ policy ≈ ref) —
   แลกความแม่นกับ VRAM บน vocab ขนาด 151,936

7. **รันครั้งเดียว ไม่ได้ทำ multiple seeds** — PPO ขึ้นชื่อเรื่องความแปรปรวนข้าม seed
   เป็นพิเศษ (นี่คือหนึ่งในเหตุผลการมีอยู่ของตอนที่ 4) ใช้ seed=42 ตายตัวเพื่อให้ทำซ้ำได้
   แต่ไม่ได้รายงานความแปรปรวน

8. **ฮาร์ดแวร์จำกัด** — ทุกอย่างถูกบีบให้รันได้บน T4 ฟรี ซึ่งแปลว่าต้องใช้ fp16 แทน bf16,
   ใช้ sdpa แทน FlashAttention-2, batch เล็ก และ sequence สั้น
   ผลบนฮาร์ดแวร์ที่ใหญ่กว่าอาจต่างออกไป

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) — ใช้ต่อได้โดยอ้างอิงที่มา ห้ามใช้เชิงพาณิชย์ และต้องเผยแพร่ต่อด้วยสัญญาเดียวกัน — [kobkrit.com](https://kobkrit.com)*
